# 02 — Évaluation du modèle**GPU recommended** but not required; evaluation on the test split runs in a few minutes on CPU.`train.py` emits three scalars. A defence needs more: which classes work, which fail, and why.This notebook produces the per-class breakdown, the confusion matrix, and the failure gallery —the evidence that turns "the model has 0.42 mAP" into an argument.**Inputs:** `best.pt` and `my_dataset/` from notebook 01.**Outputs:** `evaluation_report.json`, confusion matrix and PR curve figures for the report.

## 1 — Setup

In [ ]:
!pip install -q ultralytics==8.3.55 onnxruntime==1.20.1 \                opencv-python-headless==4.10.0.84 pandas matplotlib seabornimport os, subprocessif os.path.exists("/content/Nutrivision"):    !rm -rf /content/Nutrivisionsubprocess.run(["git","clone","--depth","1",                "https://github.com/Zen-Daitsu/Nutrivision.git",                "/content/Nutrivision"], check=True)%cd /content/Nutrivisionfrom google.colab import drivedrive.mount('/content/drive')ARTIFACTS = "/content/drive/MyDrive/Nutrivision/artifacts"!ls -lh {ARTIFACTS}

In [ ]:
import shutil, osos.makedirs("runs/nutrivision/weights", exist_ok=True)shutil.copy2(f"{ARTIFACTS}/best.pt", "runs/nutrivision/weights/best.pt")# The dataset must be present for validation. Symlink it from Drive.SRC = "/content/drive/MyDrive/Nutrivision/my_dataset"if os.path.isdir(SRC) and not os.path.exists("my_dataset"):    os.symlink(SRC, "my_dataset")!ls my_datasetprint(open("my_dataset/data.yaml").read())

## 2 — Run validation on the held-out test splitThe test split has been untouched by training and by early stopping. Numbers from thevalidation split are optimistic; these are the ones to publish.

In [ ]:
from ultralytics import YOLOmodel = YOLO("runs/nutrivision/weights/best.pt")metrics = model.val(data="my_dataset/data.yaml", split="test", plots=True)print()print(f"mask  mAP50    : {metrics.seg.map50:.4f}")print(f"mask  mAP50-95 : {metrics.seg.map:.4f}")print(f"box   mAP50    : {metrics.box.map50:.4f}")print(f"box   mAP50-95 : {metrics.box.map:.4f}")

### Per-class breakdownAn aggregate mAP hides everything that matters. A model at 0.45 overall might be 0.80 onbroccoli and 0.05 on chicken — and only the per-class table tells you which.

In [ ]:
import pandas as pd, yaml, numpy as npnames = yaml.safe_load(open("my_dataset/data.yaml"))["names"]rows = []for i, cls_idx in enumerate(metrics.seg.ap_class_index):    rows.append({        "class": names[int(cls_idx)],        "mask_AP50": round(float(metrics.seg.ap50[i]), 4),        "mask_AP50_95": round(float(metrics.seg.ap[i]), 4),        "precision": round(float(metrics.seg.p[i]), 4),        "recall": round(float(metrics.seg.r[i]), 4),    })per_class = pd.DataFrame(rows).sort_values("mask_AP50", ascending=False)per_class

In [ ]:
import matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(11, 5))colors = ["#7FD1B9" if v >= 0.5 else "#F2C14E" if v >= 0.3 else "#E0715F"          for v in per_class["mask_AP50"]]ax.barh(per_class["class"], per_class["mask_AP50"], color=colors)ax.axvline(0.5, color="#2A363D", ls="--", lw=1, label="usable threshold")ax.set_xlabel("mask AP@50")ax.set_title("Per-class segmentation performance on the test split")ax.invert_yaxis(); ax.legend()plt.tight_layout(); plt.show()weak = per_class[per_class["mask_AP50"] < 0.30]if len(weak):    print("Classes below usable threshold — do not claim these work:")    for _, r in weak.iterrows():        print(f"  {r['class']:16s} AP50 {r['mask_AP50']:.3f}")

### Confusion matrixWhich classes the model conflates is more diagnostic than the score. Rice confused withquinoa is forgivable; chicken confused with broccoli means the class map is wrong.

In [ ]:
from IPython.display import Image, displayimport globfor fig_path in sorted(glob.glob("runs/segment/**/confusion_matrix_normalized.png", recursive=True)) + \                sorted(glob.glob("runs/segment/**/MaskPR_curve.png", recursive=True)) + \                sorted(glob.glob("runs/segment/**/MaskF1_curve.png", recursive=True)):    print(fig_path)    display(Image(fig_path, width=760))

## 3 — Failure galleryTwelve worst predictions on the test split. These images are what you show when an examinerasks where the model breaks — having an answer is stronger than not having been asked.

In [ ]:
import glob, random, cv2import numpy as nptest_images = sorted(glob.glob("my_dataset/test/images/*"))print(f"{len(test_images)} test images")scored = []for path in test_images:    r = model.predict(path, verbose=False, conf=0.25)[0]    conf = float(r.boxes.conf.max()) if r.boxes is not None and len(r.boxes) else 0.0    scored.append((conf, path))scored.sort()worst = scored[:12]fig, axes = plt.subplots(3, 4, figsize=(18, 12))for ax, (conf, path) in zip(axes.ravel(), worst):    r = model.predict(path, verbose=False, conf=0.25)[0]    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))    ax.set_title(f"max conf {conf:.2f}", fontsize=9)    ax.axis("off")plt.suptitle("Twelve lowest-confidence test predictions", fontsize=14)plt.tight_layout(); plt.show()

## 4 — Export the report

In [ ]:
import json, datetimereport = {    "generated_utc": datetime.datetime.utcnow().isoformat() + "Z",    "split": "test",    "aggregate": {        "mask_map50": round(float(metrics.seg.map50), 4),        "mask_map50_95": round(float(metrics.seg.map), 4),        "box_map50": round(float(metrics.box.map50), 4),        "box_map50_95": round(float(metrics.box.map), 4),    },    "per_class": per_class.to_dict(orient="records"),    "classes_below_threshold": weak["class"].tolist() if len(weak) else [],    "n_test_images": len(test_images),}with open("evaluation_report.json", "w") as f:    json.dump(report, f, indent=2)import shutilshutil.copy2("evaluation_report.json", ARTIFACTS)per_class.to_csv(f"{ARTIFACTS}/per_class_metrics.csv", index=False)print(json.dumps(report["aggregate"], indent=2))print("\nsaved to", ARTIFACTS)

## Interpreting the result honestly| mask mAP50 | What it means | What to say ||---|---|---|| below 0.20 | The model has not learned the task | The class map or the corpus is wrong. Return to notebook 00. || 0.20 – 0.40 | Learns some classes, fails others | Report per-class. Name the weak classes explicitly. || 0.40 – 0.60 | Reasonable for a 10-class food segmenter on FoodSeg103 alone | Defensible. Note that real-world plates are harder than the benchmark. || above 0.60 | Strong for this corpus | Verify the test split was not contaminated — check the sha1 routing in `compile_dataset.py`. |FoodSeg103 is a benchmark with clean photography. Cafeteria trays under fluorescent lightare a different distribution. Whatever number you report, state that the 300 real-captureimages belong in the *training* mix, not only in evaluation.